# Model evaluation & error analysis

Load **one trained captioning checkpoint** (from Google Drive), run it over the COCO images, score **every image with its own BLEU-4**, then:

1. **Rank** all images by BLEU-4.
2. **Show the lowest-`K` and highest-`K`** examples (image + the model's caption + a human reference + the score).
3. **Compare keywords** — the `J` most common content words the model produced for its **worst** vs **best** captions, to surface what it does well/badly.

Works with any architecture trained by the other notebooks (CNN/ViT/CLIP + GPT-2, or CNN+GRU): the checkpoint stores its own `config`, so this notebook rebuilds the exact model automatically. Built for Colab (also runs on a local Jupyter).

## 1. Install dependencies and imports

In [ ]:
# Install once if needed
!pip -q install transformers pycocotools nltk

import os, json, random, time, copy, math, textwrap, shutil
from dataclasses import dataclass, asdict, fields
from typing import Optional
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision import models

from transformers import (
    AutoConfig, AutoModel, AutoModelForCausalLM, AutoTokenizer,
    AutoImageProcessor, CLIPVisionModel,
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Using device:", device)

## 2. Settings & mount Drive

Point `MODEL_PATH` at the checkpoint you saved to Drive by the training notebook (e.g. `vit_gpt2_best.pt`).

- `EVAL_SPLIT` — `"all"` scores every image in the COCO split (richer signal for error analysis, but ~90% were seen in training, so it's *qualitative*, not an unbiased metric). `"val"` restricts to the held-out validation images (the clean ~10%).
- `NUM_EVAL_IMAGES` — cap for a quick run; `None` = all images in the chosen split.
- `K` — how many lowest/highest examples to display. `J` — keywords per group. `KEYWORD_GROUP_SIZE` — how many captions feed each keyword tally.

In [ ]:
# Portable paths: works on Colab (mounts Drive) AND a local Jupyter notebook.
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content"
    ON_COLAB = True
except ImportError:
    BASE_DIR = os.path.abspath(".")
    ON_COLAB = False

DATA_DIR = os.path.join(BASE_DIR, "data", "coco")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs_eval")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- The checkpoint to evaluate (edit this) ---------------------------------
MODEL_PATH = "/content/drive/MyDrive/image_captioning_finetune/vit_gpt2_best.pt"

# ---- What to evaluate on -----------------------------------------------------
COCO_SPLIT = "val"        # which COCO image set is on disk
EVAL_SPLIT = "all"        # "all" = every image in COCO_SPLIT (~5000); "val" = held-out 10%
NUM_EVAL_IMAGES = None    # e.g. 1000 for a quick pass; None = all in EVAL_SPLIT

# ---- Inference / analysis knobs ---------------------------------------------
EVAL_BATCH_SIZE = 32
MAX_GEN_LEN = 40
MAX_TEXT_LEN = 40
NUM_WORKERS = 2

K = 8                     # show K lowest + K highest scoring examples
J = 15                    # top-J keywords per group
KEYWORD_GROUP_SIZE = 100  # captions feeding each keyword tally (worst K..., best K...)

print("Model path:", MODEL_PATH)
print("Output dir:", OUTPUT_DIR)
print("Eval split:", EVAL_SPLIT, "| cap:", NUM_EVAL_IMAGES)

## 3. Download MS-COCO data

Pure-Python download/extract; skips the work if the data is already present (reused if you ran a training notebook in the same session).

In [ ]:
import urllib.request, zipfile

ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
IMAGES_URL = ("http://images.cocodataset.org/zips/val2017.zip" if COCO_SPLIT == "val"
              else "http://images.cocodataset.org/zips/train2017.zip")

ann_dir = os.path.join(DATA_DIR, "annotations")
img_dir = os.path.join(DATA_DIR, f"{COCO_SPLIT}2017")

def _download_and_extract(url, zip_path, extract_to):
    def _progress(block_num, block_size, total_size):
        if total_size > 0:
            pct = min(100, block_num * block_size * 100 / total_size)
            print(f"\r  downloading... {pct:5.1f}%", end="")
    print("Downloading", url)
    urllib.request.urlretrieve(url, zip_path, _progress)
    print("\n  extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    os.remove(zip_path)
    print("  done.")

if not os.path.exists(ann_dir):
    _download_and_extract(ANNOTATIONS_URL, os.path.join(DATA_DIR, "annotations.zip"), DATA_DIR)
else:
    print("Annotations already extracted.")

if not os.path.exists(img_dir):
    _download_and_extract(IMAGES_URL, os.path.join(DATA_DIR, f"{COCO_SPLIT}2017.zip"), DATA_DIR)
else:
    print(f"{COCO_SPLIT}2017 images already extracted.")

## 4. Load captions and reproduce the train/val split

The **same** leakage-safe 90/10 split-by-image-id (seed 42) used in training, so `EVAL_SPLIT="val"` lines up exactly with the model's held-out set, and we can flag which images were seen during training.

In [ ]:
annotations_file = os.path.join(DATA_DIR, "annotations", f"captions_{COCO_SPLIT}2017.json")
with open(annotations_file, "r") as f:
    coco_data = json.load(f)

img_id_to_filename = {img["id"]: img["file_name"] for img in coco_data["images"]}
img_id_to_captions = defaultdict(list)
for ann in coco_data["annotations"]:
    img_id_to_captions[ann["image_id"]].append(ann["caption"])

def make_image_id_split(image_ids, train_fraction=0.90, seed=42):
    image_ids = list(image_ids)
    rng = random.Random(seed)
    rng.shuffle(image_ids)
    split_idx = int(train_fraction * len(image_ids))
    return set(image_ids[:split_idx]), set(image_ids[split_idx:])

train_img_ids, val_img_ids = make_image_id_split(img_id_to_filename.keys(), 0.90, SEED)
train_annotations = [a for a in coco_data["annotations"] if a["image_id"] in train_img_ids]

print("Total images:", len(img_id_to_filename))
print("Train images:", len(train_img_ids), " Val images:", len(val_img_ids))

## 5. Unified encoder/decoder framework

The same building blocks the training notebooks used, so a checkpoint's saved `config` rebuilds the identical model. `build_model(config)` constructs any encoder (CNN/ViT/CLIP) + decoder (GPT-2/GRU) combination.

In [ ]:
# ---- Word-level vocabulary (for GRU checkpoints) ----------------------------
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        self.word2idx = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.word_count = Counter()

    def build(self, captions):
        for cap in captions:
            self.word_count.update(word_tokenize(cap.lower()))
        idx = 4
        for w, c in self.word_count.items():
            if c >= self.freq_threshold:
                self.word2idx[w] = idx; self.idx2word[idx] = w; idx += 1

    def numericalize(self, cap):
        ids = [self.word2idx["<start>"]]
        ids += [self.word2idx.get(t, self.word2idx["<unk>"]) for t in word_tokenize(cap.lower())]
        ids.append(self.word2idx["<end>"])
        return ids

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), "<unk>")
            if w == "<end>": break
            if w not in ("<start>", "<pad>"): words.append(w)
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

rnn_vocab = Vocabulary(freq_threshold=5)
rnn_vocab.build([a["caption"] for a in train_annotations])
print("GRU word-level vocab size:", len(rnn_vocab))

# ---- Image preprocessing ----------------------------------------------------
resnet_transform = T.Compose([
    T.Resize((256, 256)), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

def preprocess_images(images, encoder_kind, image_processor):
    if encoder_kind == "cnn":
        return torch.stack([resnet_transform(im) for im in images], 0)
    return image_processor(list(images), return_tensors="pt").pixel_values

# ---- Encoder: image -> sequence of feature vectors (B, S, D_enc) ------------
class ImageEncoder(nn.Module):
    def __init__(self, kind, name):
        super().__init__()
        self.kind = kind
        if kind == "cnn":
            resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            self.backbone = nn.Sequential(*list(resnet.children())[:-2])
            self.feat_dim = 2048
        elif kind == "clip":
            self.backbone = CLIPVisionModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size
        else:  # vit
            self.backbone = AutoModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size

    def forward(self, images):
        if self.kind == "cnn":
            f = self.backbone(images)
            B, C, H, W = f.shape
            return f.view(B, C, H * W).permute(0, 2, 1)
        out = self.backbone(pixel_values=images)
        return out.last_hidden_state

# ---- Decoder A: word-level GRU ----------------------------------------------
class GRUDecoder(nn.Module):
    def __init__(self, feat_dim, vocab, embed_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.vocab = vocab
        self.pad_id = vocab.word2idx["<pad>"]
        self.end_id = vocab.word2idx["<end>"]
        self.img_proj = nn.Linear(feat_dim, embed_size)
        self.bn = nn.BatchNorm1d(embed_size)
        self.embed = nn.Embedding(len(vocab), embed_size)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, len(vocab))
        self.criterion = nn.CrossEntropyLoss(ignore_index=self.pad_id)

    def _img_token(self, enc_seq):
        pooled = enc_seq.mean(dim=1)
        return self.bn(self.img_proj(pooled))

    @torch.no_grad()
    def generate(self, enc_seq, max_len):
        feat = self._img_token(enc_seq)
        B = feat.size(0)
        inp = feat.unsqueeze(1)
        states = None
        done = torch.zeros(B, dtype=torch.bool, device=feat.device)
        seqs = [[] for _ in range(B)]
        for _ in range(max_len):
            hiddens, states = self.rnn(inp, states)
            pred = self.linear(hiddens.squeeze(1)).argmax(1)
            for i in range(B):
                if done[i]:
                    continue
                tid = int(pred[i].item())
                if tid == self.end_id:
                    done[i] = True
                else:
                    seqs[i].append(tid)
            if bool(done.all()):
                break
            inp = self.embed(pred).unsqueeze(1)
        return seqs

    def decode(self, ids):
        return self.vocab.decode(ids)

# ---- Decoder B: GPT-2 with cross-attention ----------------------------------
class GPT2Decoder(nn.Module):
    def __init__(self, feat_dim, tokenizer, dropout=0.1, freeze_base=False):
        super().__init__()
        cfg = AutoConfig.from_pretrained("gpt2")
        cfg.is_decoder = True
        cfg.add_cross_attention = True
        cfg.resid_pdrop = dropout
        cfg.embd_pdrop = dropout
        cfg.attn_pdrop = dropout
        self.gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", config=cfg)
        self.gpt2.resize_token_embeddings(len(tokenizer))
        self.enc_proj = nn.Linear(feat_dim, cfg.n_embd)
        self.tokenizer = tokenizer
        self.bos_id = tokenizer.bos_token_id
        self.eos_id = tokenizer.eos_token_id
        self.pad_id = tokenizer.pad_token_id

    @torch.no_grad()
    def generate(self, enc_seq, max_len):
        enc_hidden = self.enc_proj(enc_seq)
        B = enc_hidden.size(0)
        dev = enc_hidden.device
        cur = torch.full((B, 1), self.bos_id, dtype=torch.long, device=dev)
        seqs = [[] for _ in range(B)]
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        past = None
        for _ in range(max_len):
            out = self.gpt2(input_ids=cur, encoder_hidden_states=enc_hidden,
                            past_key_values=past, use_cache=True)
            past = out.past_key_values
            nxt = out.logits[:, -1, :].argmax(-1)
            for i in range(B):
                if done[i]:
                    continue
                tid = int(nxt[i].item())
                if tid == self.eos_id:
                    done[i] = True
                else:
                    seqs[i].append(tid)
            if bool(done.all()):
                break
            cur = nxt.unsqueeze(1)
        return seqs

    def decode(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=True).strip()

# ---- Full model -------------------------------------------------------------
class CaptioningModel(nn.Module):
    def __init__(self, encoder, decoder, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder

    def _encode(self, images):
        if self.freeze_encoder:
            with torch.no_grad():
                return self.encoder(images)
        return self.encoder(images)

    @torch.no_grad()
    def generate(self, images, max_len):
        return self.decoder.generate(self._encode(images), max_len)

    def decode(self, ids):
        return self.decoder.decode(ids)

# ---- Config + builder (mirror of the training notebooks) --------------------
@dataclass
class ExperimentConfig:
    name: str
    encoder_kind: str
    encoder_name: str
    decoder: str
    learning_rate: float
    weight_decay: float = 0.0
    dropout: float = 0.1
    freeze_gpt2_base: bool = False
    embed_size: int = 256
    hidden_size: int = 512
    num_layers: int = 2
    freq_threshold: int = 5
    batch_size: int = 16
    epochs: int = 15
    max_train_batches: Optional[int] = 300

def build_model(config):
    encoder = ImageEncoder(config.encoder_kind, config.encoder_name)
    feat_dim = encoder.feat_dim
    if config.decoder == "gru":
        decoder = GRUDecoder(feat_dim, rnn_vocab, config.embed_size,
                             config.hidden_size, config.num_layers, config.dropout)
    elif config.decoder == "gpt2":
        tok = AutoTokenizer.from_pretrained("gpt2")
        if tok.pad_token is None:
            tok.add_special_tokens({"pad_token": "<|pad|>"})
        decoder = GPT2Decoder(feat_dim, tok, dropout=config.dropout,
                              freeze_base=config.freeze_gpt2_base)
    else:
        raise ValueError(f"Unknown decoder: {config.decoder}")
    model = CaptioningModel(encoder, decoder, freeze_encoder=True)
    image_processor = (None if config.encoder_kind == "cnn"
                       else AutoImageProcessor.from_pretrained(config.encoder_name))
    return {"model": model, "encoder_kind": config.encoder_kind,
            "decoder_kind": config.decoder, "image_processor": image_processor}

# ---- Eval-only dataset: one row per unique image ----------------------------
class UniqueImageDataset(Dataset):
    def __init__(self, img_dir, image_ids, img_id_to_filename):
        self.img_dir = img_dir
        self.image_ids = list(image_ids)
        self.map = dict(img_id_to_filename)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        iid = self.image_ids[idx]
        path = os.path.join(self.img_dir, self.map[iid])
        return Image.open(path).convert("RGB"), iid

def make_eval_collate(bundle):
    kind = bundle["encoder_kind"]
    image_processor = bundle["image_processor"]
    def collate(batch):
        images, ids = zip(*batch)
        pixel_values = preprocess_images(images, kind, image_processor)
        return pixel_values, torch.tensor(ids, dtype=torch.long)
    return collate

print("Framework ready.")

## 6. Load the checkpoint

Rebuilds the exact architecture from the `config` stored inside the checkpoint, then loads the trained weights. Works for any model the training notebooks produced.

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location=device)
# Some checkpoints (e.g. the ViT-only notebook) don't store every field this
# notebook's ExperimentConfig expects. Infer/fill the missing ones.
saved_cfg = dict(ckpt["config"])
if "encoder_kind" not in saved_cfg:
    _nm = str(saved_cfg.get("encoder_name", "")).lower()
    if "clip" in _nm:
        saved_cfg["encoder_kind"] = "clip"
    elif "resnet" in _nm or _nm == "cnn":
        saved_cfg["encoder_kind"] = "cnn"
    else:
        saved_cfg["encoder_kind"] = "vit"
saved_cfg.setdefault("name", os.path.splitext(os.path.basename(MODEL_PATH))[0])
saved_cfg.setdefault("decoder", "gpt2")

valid = {f.name for f in fields(ExperimentConfig)}
cfg_dict = {k: v for k, v in saved_cfg.items() if k in valid}
config = ExperimentConfig(**cfg_dict)
MODEL_NAME = config.name

bundle = build_model(config)
model = bundle["model"].to(device)
missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)
model.eval()

print("Loaded:", MODEL_NAME)
print(f"  encoder = {config.encoder_kind} ({config.encoder_name})")
print(f"  decoder = {config.decoder} | dropout={config.dropout} | "
      f"freeze_gpt2_base={config.freeze_gpt2_base}")
print(f"  reported val BLEU-4 at save time = {ckpt.get('bleu4')}  (epoch {ckpt.get('epoch')})")
if missing:    print("  [warn] missing keys:", len(missing))
if unexpected: print("  [warn] unexpected keys:", len(unexpected))

## 7. Run inference and score every image with its own BLEU-4

For each unique image we generate one caption and compute **sentence-level BLEU-4** against *all* of that image's human references (smoothing `method1`, matching the training metric). Results are sorted and saved to CSV.

In [ ]:
# Choose which images to score.
if EVAL_SPLIT == "val":
    eval_ids = [i for i in img_id_to_filename if i in val_img_ids]
else:
    eval_ids = list(img_id_to_filename.keys())
eval_ids = sorted(eval_ids)
if NUM_EVAL_IMAGES is not None:
    eval_ids = eval_ids[:NUM_EVAL_IMAGES]

print("Scoring", len(eval_ids), "unique images (split:", EVAL_SPLIT, ")")
if EVAL_SPLIT == "all":
    seen = sum(1 for i in eval_ids if i in train_img_ids)
    print(f"  note: {seen}/{len(eval_ids)} were seen during training -> qualitative "
          f"analysis, NOT an unbiased metric. Use EVAL_SPLIT='val' for the clean subset.")

eval_ds = UniqueImageDataset(img_dir, eval_ids, img_id_to_filename)
eval_collate = make_eval_collate(bundle)
eval_loader = DataLoader(eval_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, collate_fn=eval_collate,
                         pin_memory=torch.cuda.is_available())

smoothing = SmoothingFunction().method1
records = []
done = 0
t0 = time.time()
with torch.no_grad():
    for pixel_values, ids in eval_loader:
        pixel_values = pixel_values.to(device)
        gen_ids = model.generate(pixel_values, MAX_GEN_LEN)
        for j in range(pixel_values.size(0)):
            iid = int(ids[j].item())
            hyp = model.decode(gen_ids[j])
            hyp_tok = word_tokenize(hyp.lower())
            refs = img_id_to_captions[iid]
            refs_tok = [word_tokenize(r.lower()) for r in refs]
            score = (sentence_bleu(refs_tok, hyp_tok, weights=(0.25, 0.25, 0.25, 0.25),
                                   smoothing_function=smoothing) if hyp_tok else 0.0)
            records.append({"image_id": iid, "filename": img_id_to_filename[iid],
                            "hypothesis": hyp, "ref0": refs[0],
                            "num_refs": len(refs), "bleu4": score})
        done += pixel_values.size(0)
        print(f"\r  {done}/{len(eval_ids)} images  ({time.time()-t0:.0f}s)", end="")
print()

records.sort(key=lambda r: r["bleu4"])
rec_df = pd.DataFrame(records)
scores_csv = os.path.join(OUTPUT_DIR, f"per_image_bleu_{MODEL_NAME}.csv")
rec_df.to_csv(scores_csv, index=False)

# Corpus BLEU over the whole eval set (the headline-style number).
corpus_refs = [[word_tokenize(c.lower()) for c in img_id_to_captions[r["image_id"]]] for r in records]
corpus_hyps = [word_tokenize(r["hypothesis"].lower()) for r in records]
corpus_b = corpus_bleu(corpus_refs, corpus_hyps, smoothing_function=smoothing)

print(f"mean per-image BLEU-4 = {rec_df['bleu4'].mean():.4f}")
print(f"corpus      BLEU-4 = {corpus_b:.4f}")
print("saved:", scores_csv)
rec_df.describe()[["bleu4"]]

## 8. Lowest and highest scoring captions

The `K` worst and `K` best images, each with the model's caption (`Pred`), one human reference (`Ref`), and the BLEU-4 score.

In [ ]:
def show_grid(recs, suptitle, save_path):
    n = len(recs)
    cols = min(4, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.2, rows * 4.8))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, r in zip(axes, recs):
        img = Image.open(os.path.join(img_dir, r["filename"])).convert("RGB")
        ax.imshow(img)
        pred = textwrap.fill("Pred: " + r["hypothesis"], 40)
        ref = textwrap.fill("Ref: " + r["ref0"], 40)
        ax.set_title(f"BLEU-4 = {r['bleu4']:.3f}\n{pred}\n{ref}", fontsize=8)
    fig.suptitle(suptitle, fontsize=15, y=1.0)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("saved:", save_path)

lowest = records[:K]
highest = records[-K:][::-1]

show_grid(lowest,  f"{K} LOWEST BLEU-4  -  {MODEL_NAME}",
          os.path.join(OUTPUT_DIR, f"lowest_bleu_{MODEL_NAME}.png"))
show_grid(highest, f"{K} HIGHEST BLEU-4  -  {MODEL_NAME}",
          os.path.join(OUTPUT_DIR, f"highest_bleu_{MODEL_NAME}.png"))

## 9. Keyword comparison — what the model says when it fails vs succeeds

Pool the model's captions for its **worst `KEYWORD_GROUP_SIZE`** and **best `KEYWORD_GROUP_SIZE`** images, drop stop-words, and show the `J` most common content words in each. Words that dominate the *low* group but not the *high* group hint at systematic failure modes (e.g. generic fallbacks, repeated nouns).

In [ ]:
STOP = set(stopwords.words("english"))

def top_keywords(recs, j):
    c = Counter()
    for r in recs:
        for t in word_tokenize(r["hypothesis"].lower()):
            if t.isalpha() and len(t) > 2 and t not in STOP:
                c[t] += 1
    return c.most_common(j)

g = min(KEYWORD_GROUP_SIZE, len(records))
low_group = records[:g]
high_group = records[-g:]
low_kw = top_keywords(low_group, J)
high_kw = top_keywords(high_group, J)

print(f"Worst {g} captions  -> top {J} keywords:")
print("  ", ", ".join(f"{w}({c})" for w, c in low_kw))
print(f"Best  {g} captions  -> top {J} keywords:")
print("  ", ", ".join(f"{w}({c})" for w, c in high_kw))

low_set = {w for w, _ in low_kw}
high_set = {w for w, _ in high_kw}
print("\nOnly in WORST group:", sorted(low_set - high_set))
print("Only in BEST  group:", sorted(high_set - low_set))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, kw, title in [(axes[0], low_kw, f"Worst {g} captions"),
                      (axes[1], high_kw, f"Best {g} captions")]:
    words = [w for w, _ in kw][::-1]
    counts = [c for _, c in kw][::-1]
    ax.barh(words, counts)
    ax.set_title(f"Top {J} keywords - {title}")
    ax.set_xlabel("count")
fig.suptitle(f"Keyword comparison  -  {MODEL_NAME}", fontsize=14)
fig.tight_layout()
kw_png = os.path.join(OUTPUT_DIR, f"keyword_comparison_{MODEL_NAME}.png")
fig.savefig(kw_png, dpi=150, bbox_inches="tight")
plt.show()

kw_csv = os.path.join(OUTPUT_DIR, f"keyword_comparison_{MODEL_NAME}.csv")
pd.DataFrame({
    "low_keyword":  [w for w, _ in low_kw],  "low_count":  [c for _, c in low_kw],
    "high_keyword": [w for w, _ in high_kw], "high_count": [c for _, c in high_kw],
}).to_csv(kw_csv, index=False)
print("saved:", kw_png)
print("saved:", kw_csv)

## 10. Save analysis outputs to Drive (optional)

Copies the per-image scores, example grids, and keyword comparison to Drive so they survive the Colab VM.

In [ ]:
ANALYSIS_DRIVE_DIR = "/content/drive/MyDrive/image_captioning_eval"
out_files = [
    scores_csv,
    os.path.join(OUTPUT_DIR, f"lowest_bleu_{MODEL_NAME}.png"),
    os.path.join(OUTPUT_DIR, f"highest_bleu_{MODEL_NAME}.png"),
    kw_png, kw_csv,
]
if ON_COLAB:
    os.makedirs(ANALYSIS_DRIVE_DIR, exist_ok=True)
    for f in out_files:
        if os.path.exists(f):
            shutil.copy2(f, os.path.join(ANALYSIS_DRIVE_DIR, os.path.basename(f)))
    print("Copied analysis outputs to Drive:", ANALYSIS_DRIVE_DIR)
else:
    print("Not on Colab - outputs persist locally under:", os.path.abspath(OUTPUT_DIR))